In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
Firms = "/content/drive/MyDrive/India_Firms"
import pandas as pd

In [3]:
df = pd.read_parquet(f"{Firms}/India_Firms.parquet")

In [9]:
from pathlib import Path
import gc
import os
import json
import pandas as pd
import numpy as np

# ============================================================
# PRODUCTION CONFIGURATION
# ============================================================

SPATIAL_THRESHOLD_M = 1000.0
TEMPORAL_THRESHOLD_HOURS = 12.0
TEMPORAL_THRESHOLD_SECONDS = 12 * 3600

PROJ_EPSG = 6933

# Complete observed period
START_DATE = pd.Timestamp("2012-01-20")
END_DATE   = pd.Timestamp("2026-08-31")

MISSING_DATES = {
    pd.Timestamp("2012-03-25").date()
}

# ------------------------------------------------------------
# CHANGE THESE PATHS TO YOUR ACTUAL PROJECT DIRECTORIES
# ------------------------------------------------------------

PROJECT_ROOT = Path("fire_event_project")

CLUSTER_DIR = PROJECT_ROOT / "clusters"
EDGE_DIR = PROJECT_ROOT / "edges"
GLOBAL_DIR = PROJECT_ROOT / "global"
QC_DIR = PROJECT_ROOT / "qc"
LOG_DIR = PROJECT_ROOT / "logs"

for d in [
    CLUSTER_DIR,
    EDGE_DIR,
    GLOBAL_DIR,
    QC_DIR,
    LOG_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

print("Production directories ready.")
print("Cluster directory:", CLUSTER_DIR)
print("Edge directory:", EDGE_DIR)
print("Global directory:", GLOBAL_DIR)
print("QC directory:", QC_DIR)


Production directories ready.
Cluster directory: fire_event_project/clusters
Edge directory: fire_event_project/edges
Global directory: fire_event_project/global
QC directory: fire_event_project/qc


In [10]:
# ============================================================
# OBSERVED DATE INDEX
# ============================================================

all_calendar_dates = pd.date_range(
    START_DATE,
    END_DATE,
    freq="D"
)

observed_dates = [
    d
    for d in all_calendar_dates
    if d.date() not in MISSING_DATES
]

print("Calendar dates:", len(all_calendar_dates))
print("Observed dates:", len(observed_dates))
print("Missing dates:", len(all_calendar_dates) - len(observed_dates))

print("\nFirst 5:")
print(observed_dates[:5])

print("\nLast 5:")
print(observed_dates[-5:])

assert len(observed_dates) == 5337
assert pd.Timestamp("2012-03-25") not in observed_dates

print("\n✅ Date-index validation passed")


Calendar dates: 5338
Observed dates: 5337
Missing dates: 1

First 5:
[Timestamp('2012-01-20 00:00:00'), Timestamp('2012-01-21 00:00:00'), Timestamp('2012-01-22 00:00:00'), Timestamp('2012-01-23 00:00:00'), Timestamp('2012-01-24 00:00:00')]

Last 5:
[Timestamp('2026-08-27 00:00:00'), Timestamp('2026-08-28 00:00:00'), Timestamp('2026-08-29 00:00:00'), Timestamp('2026-08-30 00:00:00'), Timestamp('2026-08-31 00:00:00')]

✅ Date-index validation passed


In [8]:
from pathlib import Path

print("=" * 70)
print("SEARCHING FOR FIRE-EVENT PRODUCTION OUTPUT")
print("=" * 70)

base = Path("/content")

matches = []

for p in base.rglob("*"):
    if p.is_file() and (
        p.name.startswith("clusters_") or
        p.name.startswith("edges_") or
        "global_cluster_to_event" in p.name
    ):
        matches.append(p)

print(f"Files found: {len(matches):,}")

for p in sorted(matches)[:30]:
    print(p)

if len(matches) > 30:
    print(f"... and {len(matches)-30:,} more")


SEARCHING FOR FIRE-EVENT PRODUCTION OUTPUT
Files found: 0


In [14]:
# ============================================================
# SOURCE DATA VALIDATION
# ============================================================

required_columns = [
    "latitude",
    "longitude",
    "datetime",
    "date",
    "x",
    "y",
    "satellite",
    "frp"
]

missing_columns = [
    c for c in required_columns
    if c not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nDate range:")
print(df["date"].min())
print(df["date"].max())

# Coordinate QC
bad_lat = (
    df["latitude"].isna()
    | ~df["latitude"].between(-90, 90)
)

bad_lon = (
    df["longitude"].isna()
    | ~df["longitude"].between(-180, 180)
)

bad_datetime = df["datetime"].isna()

print("\nCoordinate/datetime QC:")
print("Bad latitude :", int(bad_lat.sum()))
print("Bad longitude:", int(bad_lon.sum()))
print("Bad datetime :", int(bad_datetime.sum()))

assert bad_lat.sum() == 0
assert bad_lon.sum() == 0
assert bad_datetime.sum() == 0

print("\n✅ Source-data QC passed")


Rows: 15392720
Columns: 22

Date range:
2012-01-20
2026-08-31

Coordinate/datetime QC:
Bad latitude : 0
Bad longitude: 0
Bad datetime : 0

✅ Source-data QC passed


In [16]:
# ============================================================
# COMPACT DATE INDEX
# ============================================================

date_index = {}

for date_value, group in df.groupby("date", sort=False):
    date_index[pd.Timestamp(date_value).date()] = group.index.to_numpy()

print("Indexed dates:", len(date_index))

assert len(date_index) == 5337
assert pd.Timestamp("2012-03-25").date() not in date_index

print("✅ Date index ready")


Indexed dates: 5337
✅ Date index ready


In [17]:
# ============================================================
# PROGRESS LOG
# ============================================================

progress_file = production_log_path()

if progress_file.exists():
    progress_df = pd.read_csv(progress_file)
else:
    progress_df = pd.DataFrame(
        columns=[
            "date",
            "cluster_file",
            "cluster_rows",
            "edge_file",
            "edge_rows",
            "status"
        ]
    )

print(progress_df.tail())


NameError: name 'production_log_path' is not defined

In [18]:
from pathlib import Path

# Search the current working directory for existing production outputs
cwd = Path.cwd()

print("Current working directory:")
print(cwd)

print("\nCandidate directories:")

for p in sorted(cwd.iterdir()):
    if p.is_dir():
        print("DIR :", p)

print("\nCandidate Parquet files:")

parquet_files = list(cwd.rglob("*.parquet"))

print("Total parquet files found:", len(parquet_files))

for p in parquet_files[:30]:
    print(p)


Current working directory:
/content

Candidate directories:
DIR : /content/.config
DIR : /content/drive
DIR : /content/fire_event_production
DIR : /content/fire_event_project
DIR : /content/sample_data

Candidate Parquet files:
Total parquet files found: 1
/content/drive/MyDrive/India_Firms/India_Firms.parquet


In [19]:
from pathlib import Path

cwd = Path.cwd()

cluster_candidates = []
edge_candidates = []

for p in cwd.rglob("*.parquet"):

    name = p.name.lower()

    if "cluster" in name:
        cluster_candidates.append(p)

    if "edge" in name:
        edge_candidates.append(p)

print("CLUSTER CANDIDATES:", len(cluster_candidates))

for p in cluster_candidates[:50]:
    print(p)

print("\nEDGE CANDIDATES:", len(edge_candidates))

for p in edge_candidates[:50]:
    print(p)


CLUSTER CANDIDATES: 0

EDGE CANDIDATES: 0


In [20]:
from pathlib import Path

production_root = Path("/content/fire_event_production")
project_root = Path("/content/fire_event_project")

print("=" * 70)
print("FIRE EVENT PRODUCTION DIRECTORY")
print("=" * 70)

for root in [production_root, project_root]:

    print(f"\nROOT: {root}")

    if not root.exists():
        print("  ❌ Does not exist")
        continue

    files = list(root.rglob("*"))

    print(f"  Total items: {len(files)}")

    for p in sorted(files)[:100]:
        if p.is_dir():
            print("  DIR :", p.relative_to(root))
        else:
            print("  FILE:", p.relative_to(root))


FIRE EVENT PRODUCTION DIRECTORY

ROOT: /content/fire_event_production
  Total items: 6
  DIR : full_2012_2026
  DIR : full_2012_2026/clusters
  DIR : full_2012_2026/edges
  DIR : full_2012_2026/events
  DIR : full_2012_2026/logs
  DIR : full_2012_2026/mappings

ROOT: /content/fire_event_project
  Total items: 5
  DIR : clusters
  DIR : edges
  DIR : global
  DIR : logs
  DIR : qc


In [21]:
from pathlib import Path

search_roots = [
    Path("/content/fire_event_production"),
    Path("/content/fire_event_project"),
    Path("/content/drive/MyDrive")
]

patterns = [
    "clusters_*.parquet",
    "edges_*.parquet",
    "*cluster*.parquet",
    "*edge*.parquet",
    "*2021*.parquet"
]

for root in search_roots:

    print("\n" + "=" * 70)
    print("SEARCH ROOT:", root)
    print("=" * 70)

    if not root.exists():
        print("Does not exist")
        continue

    found = set()

    for pattern in patterns:
        for p in root.rglob(pattern):
            found.add(p)

    found = sorted(found)

    print("Files found:", len(found))

    for p in found[:100]:
        print(p)



SEARCH ROOT: /content/fire_event_production
Files found: 0

SEARCH ROOT: /content/fire_event_project
Files found: 0

SEARCH ROOT: /content/drive/MyDrive
Files found: 0


In [22]:
from pathlib import Path

# ============================================================
# FULL 2012–2026 PRODUCTION PATHS
# ============================================================

PRODUCTION_ROOT = Path(
    "/content/fire_event_production/full_2012_2026"
)

CLUSTER_DIR = PRODUCTION_ROOT / "clusters"
EDGE_DIR    = PRODUCTION_ROOT / "edges"
EVENT_DIR   = PRODUCTION_ROOT / "events"
LOG_DIR     = PRODUCTION_ROOT / "logs"
MAPPING_DIR = PRODUCTION_ROOT / "mappings"

for d in [
    CLUSTER_DIR,
    EDGE_DIR,
    EVENT_DIR,
    LOG_DIR,
    MAPPING_DIR
]:
    d.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 70)
print("FULL PRODUCTION PATHS")
print("=" * 70)

print("PRODUCTION_ROOT:", PRODUCTION_ROOT)
print("CLUSTER_DIR    :", CLUSTER_DIR)
print("EDGE_DIR       :", EDGE_DIR)
print("EVENT_DIR      :", EVENT_DIR)
print("LOG_DIR        :", LOG_DIR)
print("MAPPING_DIR    :", MAPPING_DIR)


FULL PRODUCTION PATHS
PRODUCTION_ROOT: /content/fire_event_production/full_2012_2026
CLUSTER_DIR    : /content/fire_event_production/full_2012_2026/clusters
EDGE_DIR       : /content/fire_event_production/full_2012_2026/edges
EVENT_DIR      : /content/fire_event_production/full_2012_2026/events
LOG_DIR        : /content/fire_event_production/full_2012_2026/logs
MAPPING_DIR    : /content/fire_event_production/full_2012_2026/mappings


In [23]:
# ============================================================
# PRODUCTION PROGRESS LOG
# ============================================================

PROGRESS_FILE = LOG_DIR / "production_progress.csv"


def production_log_path():
    return PROGRESS_FILE


if PROGRESS_FILE.exists():

    progress_df = pd.read_csv(
        PROGRESS_FILE
    )

    print(
        "Existing progress records:",
        len(progress_df)
    )

else:

    progress_df = pd.DataFrame(
        columns=[
            "date",
            "cluster_file",
            "cluster_rows",
            "edge_file",
            "edge_rows",
            "status"
        ]
    )

    print("No existing progress log.")


No existing progress log.


In [24]:
# ============================================================
# PRODUCTION OUTPUT INVENTORY
# ============================================================

def count_files(directory):
    return len([
        p for p in directory.rglob("*")
        if p.is_file()
    ])


print("Cluster files:", count_files(CLUSTER_DIR))
print("Edge files   :", count_files(EDGE_DIR))
print("Event files  :", count_files(EVENT_DIR))
print("Mapping files:", count_files(MAPPING_DIR))
print("Log files    :", count_files(LOG_DIR))


Cluster files: 0
Edge files   : 0
Event files  : 0
Mapping files: 0
Log files    : 0


In [25]:
import inspect

all_functions = {
    name: obj
    for name, obj in globals().items()
    if inspect.isfunction(obj)
}

keywords = [
    "cluster",
    "process",
    "cross",
    "boundary",
    "edge",
    "temporal",
    "daily",
    "production",
    "event",
    "union",
    "global"
]

print("=" * 70)
print("VALIDATED / RELEVANT FUNCTIONS")
print("=" * 70)

for name in sorted(all_functions):

    if any(
        keyword in name.lower()
        for keyword in keywords
    ):

        try:
            signature = inspect.signature(
                all_functions[name]
            )
        except Exception:
            signature = "(signature unavailable)"

        print(
            f"{name}{signature}"
        )


VALIDATED / RELEVANT FUNCTIONS
production_log_path()


In [26]:
# ============================================================
# IMPORTANT EXISTING VARIABLES
# ============================================================

interesting = [
    name
    for name in globals()
    if any(
        keyword in name.lower()
        for keyword in [
            "cluster",
            "edge",
            "boundary",
            "production",
            "event",
            "output",
            "path",
            "threshold"
        ]
    )
]

print("=" * 70)
print("RELEVANT VARIABLES")
print("=" * 70)

for name in sorted(interesting):
    print(name)


RELEVANT VARIABLES
CLUSTER_DIR
EDGE_DIR
EVENT_DIR
PRODUCTION_ROOT
Path
SPATIAL_THRESHOLD_M
TEMPORAL_THRESHOLD_HOURS
TEMPORAL_THRESHOLD_SECONDS
cluster_2021_files
cluster_candidates
edge_2021_files
edge_candidates
production_log_path
production_root


In [27]:
from pathlib import Path

# ============================================================
# SEARCH FOR SAVED PROJECT CODE
# ============================================================

search_roots = [
    Path("/content"),
    Path("/content/drive/MyDrive")
]

extensions = {
    ".py",
    ".ipynb",
    ".txt"
}

matches = []

for root in search_roots:

    if not root.exists():
        continue

    for p in root.rglob("*"):

        if p.is_file() and p.suffix.lower() in extensions:

            name = p.name.lower()

            if any(
                key in name
                for key in [
                    "fire",
                    "cluster",
                    "event",
                    "production",
                    "temporal",
                    "viirs"
                ]
            ):
                matches.append(p)

print("=" * 70)
print("POSSIBLE PROJECT CODE FILES")
print("=" * 70)

for p in sorted(set(matches)):
    print(p)

print("\nTotal:", len(set(matches)))


POSSIBLE PROJECT CODE FILES

Total: 0


In [28]:
from pathlib import Path

for root in [
    Path("/content/fire_event_production"),
    Path("/content/fire_event_project")
]:

    print("\n" + "=" * 70)
    print(root)
    print("=" * 70)

    for p in sorted(root.rglob("*")):

        relative = p.relative_to(root)

        if p.is_file():
            print(
                f"FILE  {relative} "
                f"({p.stat().st_size / 1024**2:.3f} MB)"
            )
        else:
            print(f"DIR   {relative}")



/content/fire_event_production
DIR   full_2012_2026
DIR   full_2012_2026/clusters
DIR   full_2012_2026/edges
DIR   full_2012_2026/events
DIR   full_2012_2026/logs
DIR   full_2012_2026/mappings

/content/fire_event_project
DIR   clusters
DIR   edges
DIR   global
DIR   logs
DIR   qc


In [29]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

search_patterns = [
    "*2021*.parquet",
    "*cluster*.parquet",
    "*edge*.parquet",
    "*global*.parquet",
    "*event*.parquet"
]

found = set()

for pattern in search_patterns:

    for p in drive_root.rglob(pattern):
        found.add(p)

print("=" * 70)
print("DRIVE SEARCH RESULTS")
print("=" * 70)

for p in sorted(found):
    print(p)

print("\nTotal files:", len(found))


DRIVE SEARCH RESULTS

Total files: 0


In [30]:
# ============================================================
# INSPECT NOTEBOOK EXECUTION HISTORY
# ============================================================

try:

    history = In

    print("Number of stored input cells:", len(history))

    print("\nCells containing likely production code:")

    keywords = [
        "cKDTree",
        "connected_components",
        "UnionFind",
        "union",
        "cross_day",
        "local_cluster",
        "process_temporal",
        "parquet"
    ]

    for i, cell in enumerate(history):

        if any(
            keyword in cell
            for keyword in keywords
        ):
            print("\n" + "-" * 70)
            print(f"CELL {i}")
            print("-" * 70)
            print(cell[:3000])

except Exception as e:

    print("Could not inspect notebook history:")
    print(e)


Number of stored input cells: 31

Cells containing likely production code:

----------------------------------------------------------------------
CELL 3
----------------------------------------------------------------------
df = pd.read_parquet(f"{Firms}/India_Firms.parquet")

----------------------------------------------------------------------
CELL 5
----------------------------------------------------------------------
YEAR_2021_DIR = Path(
    "/content/fire_event_production/2021_full_year"
)

cluster_2021_files = sorted(
    YEAR_2021_DIR.glob("clusters_*.parquet")
)

edge_2021_files = sorted(
    YEAR_2021_DIR.glob("edges_*.parquet")
)

print("2021 cluster files:", len(cluster_2021_files))
print("2021 edge files:", len(edge_2021_files))

----------------------------------------------------------------------
CELL 6
----------------------------------------------------------------------
from pathlib import Path

BASE_DIR = Path("fire_event_production")
YEAR_DIR = BASE_DIR / "2021_

Abi KUch Production ka start **hua**

In [31]:
# ============================================================
# FIRE-EVENT DAILY CLUSTERING ENGINE
# ============================================================

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components


# ------------------------------------------------------------
# VALIDATED THRESHOLDS
# ------------------------------------------------------------

SPATIAL_THRESHOLD_M = 1000.0
TEMPORAL_THRESHOLD_HOURS = 12.0
TEMPORAL_THRESHOLD_SECONDS = (
    TEMPORAL_THRESHOLD_HOURS * 3600.0
)


def cluster_single_day(
    day_df,
    spatial_threshold_m=SPATIAL_THRESHOLD_M,
    temporal_threshold_seconds=TEMPORAL_THRESHOLD_SECONDS
):
    """
    Cluster detections within one calendar day.

    Direct edge criterion:

        Euclidean distance <= spatial_threshold_m
        AND
        absolute time difference <= temporal_threshold_seconds

    Final clusters are connected components of this graph.
    """

    if day_df.empty:
        return (
            pd.DataFrame(),
            pd.DataFrame()
        )

    work = day_df.copy()

    # --------------------------------------------------------
    # Required fields
    # --------------------------------------------------------

    required = [
        "datetime",
        "x",
        "y",
        "frp",
        "latitude",
        "longitude",
        "satellite"
    ]

    missing = [
        c for c in required
        if c not in work.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )

    # --------------------------------------------------------
    # Stable local row positions
    # --------------------------------------------------------

    work = work.reset_index(
        drop=False
    ).rename(
        columns={"index": "_original_index"}
    )

    n = len(work)

    if n == 1:

        labels = np.zeros(
            1,
            dtype=np.int64
        )

    else:

        xy = work[
            ["x", "y"]
        ].to_numpy(
            dtype=np.float64
        )

        # ----------------------------------------------------
        # Spatial candidate search
        # ----------------------------------------------------

        tree = cKDTree(xy)

        pairs = tree.query_pairs(
            r=spatial_threshold_m,
            output_type="ndarray"
        )

        if pairs.size == 0:

            labels = np.arange(
                n,
                dtype=np.int64
            )

        else:

            pairs = np.asarray(
                pairs,
                dtype=np.int64
            ).reshape(
                -1,
                2
            )

            # ------------------------------------------------
            # Temporal filtering
            # ------------------------------------------------

            times = (
                work["datetime"]
                .astype("int64")
                .to_numpy()
            )

            dt_seconds = np.abs(
                times[pairs[:, 0]]
                -
                times[pairs[:, 1]]
            ) / 1_000_000_000.0

            valid = (
                dt_seconds
                <= temporal_threshold_seconds
            )

            valid_pairs = pairs[valid]

            if len(valid_pairs) == 0:

                labels = np.arange(
                    n,
                    dtype=np.int64
                )

            else:

                # --------------------------------------------
                # Sparse graph
                # --------------------------------------------

                rows = np.concatenate([
                    valid_pairs[:, 0],
                    valid_pairs[:, 1]
                ])

                cols = np.concatenate([
                    valid_pairs[:, 1],
                    valid_pairs[:, 0]
                ])

                data = np.ones(
                    len(rows),
                    dtype=np.uint8
                )

                graph = coo_matrix(
                    (
                        data,
                        (rows, cols)
                    ),
                    shape=(n, n)
                ).tocsr()

                n_components, labels = (
                    connected_components(
                        graph,
                        directed=False,
                        return_labels=True
                    )
                )

    # --------------------------------------------------------
    # Globally unique local cluster IDs
    # --------------------------------------------------------

    day_value = pd.Timestamp(
        work["datetime"].min()
    ).strftime("%Y%m%d")

    work["local_cluster_id"] = [
        f"{day_value}_{int(label)}"
        for label in labels
    ]

    # --------------------------------------------------------
    # Cluster statistics
    # --------------------------------------------------------

    summary_rows = []

    for cluster_id, g in work.groupby(
        "local_cluster_id",
        sort=False
    ):

        start_time = g["datetime"].min()
        end_time = g["datetime"].max()

        cx = g["x"].mean()
        cy = g["y"].mean()

        # Maximum radial distance from centroid
        dx = (
            g["x"].to_numpy(
                dtype=np.float64
            )
            - cx
        )

        dy = (
            g["y"].to_numpy(
                dtype=np.float64
            )
            - cy
        )

        max_radius_m = np.sqrt(
            dx * dx + dy * dy
        ).max()

        satellites = sorted(
            g["satellite"]
            .astype(str)
            .unique()
            .tolist()
        )

        summary_rows.append({

            "local_cluster_id":
                cluster_id,

            "start_time":
                start_time,

            "end_time":
                end_time,

            "n_detections":
                len(g),

            "centroid_x":
                cx,

            "centroid_y":
                cy,

            "total_frp":
                g["frp"].sum(),

            "mean_frp":
                g["frp"].mean(),

            "max_frp":
                g["frp"].max(),

            "n_observation_times":
                g["datetime"].nunique(),

            "duration_hours":
                (
                    end_time - start_time
                ).total_seconds() / 3600.0,

            "satellites":
                ",".join(satellites),

            "n_satellites":
                len(satellites),

            "max_radius_m":
                max_radius_m,

            "max_radius_km":
                max_radius_m / 1000.0
        })

    summary = pd.DataFrame(
        summary_rows
    )

    # --------------------------------------------------------
    # Convert centroid back to geographic coordinates
    # --------------------------------------------------------

    # EPSG:6933 -> EPSG:4326
    try:

        import geopandas as gpd
        from shapely.geometry import Point

        points = gpd.GeoSeries(
            [
                Point(x, y)
                for x, y in zip(
                    summary["centroid_x"],
                    summary["centroid_y"]
                )
            ],
            crs="EPSG:6933"
        )

        geographic = points.to_crs(
            "EPSG:4326"
        )

        summary["centroid_lon"] = (
            geographic.x.to_numpy()
        )

        summary["centroid_lat"] = (
            geographic.y.to_numpy()
        )

    except Exception as e:

        raise RuntimeError(
            "Centroid transformation failed"
        ) from e

    return work, summary


In [32]:
# ============================================================
# REGRESSION TEST — 2021-03-31
# ============================================================

TEST_DATE = pd.Timestamp(
    "2021-03-31"
).date()

test_idx = date_index[TEST_DATE]

test_day = df.loc[test_idx].copy()

print("=" * 70)
print("DAILY REGRESSION TEST")
print("=" * 70)

print("Date:", TEST_DATE)
print("Detections:", len(test_day))

test_detection, test_clusters = (
    cluster_single_day(test_day)
)

print()
print("RESULT")
print("-" * 70)
print(
    "Detections:",
    len(test_detection)
)

print(
    "Local clusters:",
    len(test_clusters)
)


DAILY REGRESSION TEST
Date: 2021-03-31
Detections: 37801

RESULT
----------------------------------------------------------------------
Detections: 37801
Local clusters: 12641


In [33]:
# ============================================================
# CLUSTER STATISTICS QC
# ============================================================

print("=" * 70)
print("CLUSTER SUMMARY QC")
print("=" * 70)

print(
    "Rows:",
    len(test_clusters)
)

print(
    "Columns:",
    list(test_clusters.columns)
)

print(
    "\nDuplicate IDs:",
    test_clusters["local_cluster_id"].duplicated().sum()
)

print(
    "Null values:",
    test_clusters.isna().sum().sum()
)

print(
    "\nDuration minimum:",
    test_clusters["duration_hours"].min()
)

print(
    "Duration maximum:",
    test_clusters["duration_hours"].max()
)

print(
    "\nRadius maximum km:",
    test_clusters["max_radius_km"].max()
)


CLUSTER SUMMARY QC
Rows: 12641
Columns: ['local_cluster_id', 'start_time', 'end_time', 'n_detections', 'centroid_x', 'centroid_y', 'total_frp', 'mean_frp', 'max_frp', 'n_observation_times', 'duration_hours', 'satellites', 'n_satellites', 'max_radius_m', 'max_radius_km', 'centroid_lon', 'centroid_lat']

Duplicate IDs: 0
Null values: 0

Duration minimum: 0.0
Duration maximum: 14.166666666666666

Radius maximum km: 5.440171076202941


In [34]:
# ============================================================
# INSTRUMENTED DAILY FIRE-EVENT CLUSTERING
# ============================================================

def cluster_single_day(
    day_df,
    spatial_threshold_m=SPATIAL_THRESHOLD_M,
    temporal_threshold_seconds=TEMPORAL_THRESHOLD_SECONDS
):
    """
    Daily fire-event clustering.

    Direct edge:
        distance <= 1,000 m
        AND
        |time difference| <= 12 h

    Final local clusters:
        connected components of the valid-edge graph.
    """

    if day_df.empty:
        return pd.DataFrame(), pd.DataFrame(), {
            "n_detections": 0,
            "n_spatial_pairs": 0,
            "n_valid_pairs": 0,
            "n_clusters": 0
        }

    work = day_df.copy()

    required = [
        "datetime",
        "x",
        "y",
        "frp",
        "latitude",
        "longitude",
        "satellite"
    ]

    missing = [
        c for c in required
        if c not in work.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )

    work = work.reset_index(
        drop=False
    ).rename(
        columns={"index": "_original_index"}
    )

    n = len(work)

    # --------------------------------------------------------
    # Spatial search
    # --------------------------------------------------------

    xy = work[
        ["x", "y"]
    ].to_numpy(dtype=np.float64)

    tree = cKDTree(xy)

    if n > 1:

        pairs = tree.query_pairs(
            r=spatial_threshold_m,
            output_type="ndarray"
        )

        pairs = np.asarray(
            pairs,
            dtype=np.int64
        ).reshape(-1, 2)

    else:

        pairs = np.empty(
            (0, 2),
            dtype=np.int64
        )

    n_spatial_pairs = len(pairs)

    # --------------------------------------------------------
    # Temporal filtering
    # --------------------------------------------------------

    if n_spatial_pairs > 0:

        times = (
            work["datetime"]
            .astype("int64")
            .to_numpy()
        )

        dt_seconds = np.abs(
            times[pairs[:, 0]]
            -
            times[pairs[:, 1]]
        ) / 1_000_000_000.0

        valid_pairs = pairs[
            dt_seconds <= temporal_threshold_seconds
        ]

    else:

        valid_pairs = np.empty(
            (0, 2),
            dtype=np.int64
        )

    n_valid_pairs = len(valid_pairs)

    # --------------------------------------------------------
    # Connected components
    # --------------------------------------------------------

    if n == 1:

        labels = np.zeros(
            1,
            dtype=np.int64
        )

    elif n_valid_pairs == 0:

        labels = np.arange(
            n,
            dtype=np.int64
        )

    else:

        rows = np.concatenate([
            valid_pairs[:, 0],
            valid_pairs[:, 1]
        ])

        cols = np.concatenate([
            valid_pairs[:, 1],
            valid_pairs[:, 0]
        ])

        data = np.ones(
            len(rows),
            dtype=np.uint8
        )

        graph = coo_matrix(
            (
                data,
                (rows, cols)
            ),
            shape=(n, n)
        ).tocsr()

        _, labels = connected_components(
            graph,
            directed=False,
            return_labels=True
        )

    # --------------------------------------------------------
    # Unique local cluster ID
    # --------------------------------------------------------

    day_value = pd.Timestamp(
        work["datetime"].min()
    ).strftime("%Y%m%d")

    work["local_cluster_id"] = [
        f"{day_value}_{int(label)}"
        for label in labels
    ]

    # --------------------------------------------------------
    # Cluster statistics
    # --------------------------------------------------------

    summary_rows = []

    for cluster_id, g in work.groupby(
        "local_cluster_id",
        sort=False
    ):

        start_time = g["datetime"].min()
        end_time = g["datetime"].max()

        cx = g["x"].mean()
        cy = g["y"].mean()

        dx = (
            g["x"].to_numpy(dtype=np.float64)
            - cx
        )

        dy = (
            g["y"].to_numpy(dtype=np.float64)
            - cy
        )

        max_radius_m = np.sqrt(
            dx * dx + dy * dy
        ).max()

        satellites = sorted(
            g["satellite"]
            .astype(str)
            .unique()
            .tolist()
        )

        summary_rows.append({
            "local_cluster_id": cluster_id,
            "start_time": start_time,
            "end_time": end_time,
            "n_detections": len(g),
            "centroid_x": cx,
            "centroid_y": cy,
            "total_frp": g["frp"].sum(),
            "mean_frp": g["frp"].mean(),
            "max_frp": g["frp"].max(),
            "n_observation_times": g["datetime"].nunique(),
            "duration_hours": (
                end_time - start_time
            ).total_seconds() / 3600.0,
            "satellites": ",".join(satellites),
            "n_satellites": len(satellites),
            "max_radius_m": max_radius_m,
            "max_radius_km": max_radius_m / 1000.0
        })

    summary = pd.DataFrame(
        summary_rows
    )

    # --------------------------------------------------------
    # Projected centroid -> WGS84
    # --------------------------------------------------------

    import geopandas as gpd
    from shapely.geometry import Point

    points = gpd.GeoSeries(
        [
            Point(x, y)
            for x, y in zip(
                summary["centroid_x"],
                summary["centroid_y"]
            )
        ],
        crs="EPSG:6933"
    )

    geographic = points.to_crs(
        "EPSG:4326"
    )

    summary["centroid_lon"] = (
        geographic.x.to_numpy()
    )

    summary["centroid_lat"] = (
        geographic.y.to_numpy()
    )

    diagnostics = {
        "n_detections": n,
        "n_spatial_pairs": n_spatial_pairs,
        "n_valid_pairs": n_valid_pairs,
        "n_clusters": len(summary)
    }

    return work, summary, diagnostics


In [35]:
# ============================================================
# MARCH 31 FULL GRAPH REGRESSION
# ============================================================

TEST_DATE = pd.Timestamp(
    "2021-03-31"
).date()

test_day = df.loc[
    date_index[TEST_DATE]
].copy()

test_detection, test_clusters, diag = (
    cluster_single_day(test_day)
)

print("=" * 70)
print("MARCH 31, 2021 — FULL REGRESSION")
print("=" * 70)

print("Detections              :", diag["n_detections"])
print("Spatial candidate pairs :", diag["n_spatial_pairs"])
print("Valid temporal pairs    :", diag["n_valid_pairs"])
print("Local clusters          :", diag["n_clusters"])

print("\nEXPECTED")
print("-" * 70)

print("Detections              : 37801")
print("Spatial candidate pairs : 89322")
print("Valid temporal pairs    : 75626")
print("Local clusters          : 12641")

print("\nREGRESSION")
print("-" * 70)

assert diag["n_detections"] == 37801
assert diag["n_spatial_pairs"] == 89322
assert diag["n_valid_pairs"] == 75626
assert diag["n_clusters"] == 12641

print("✅ ALL FOUR MARCH 31 BENCHMARKS PASSED")


MARCH 31, 2021 — FULL REGRESSION
Detections              : 37801
Spatial candidate pairs : 89322
Valid temporal pairs    : 75626
Local clusters          : 12641

EXPECTED
----------------------------------------------------------------------
Detections              : 37801
Spatial candidate pairs : 89322
Valid temporal pairs    : 75626
Local clusters          : 12641

REGRESSION
----------------------------------------------------------------------
✅ ALL FOUR MARCH 31 BENCHMARKS PASSED


In [36]:
# ============================================================
# CROSS-DAY DETECTION MATCHING
# ============================================================

def cross_day_detection_pairs(
    previous_day_df,
    current_day_df,
    spatial_threshold_m=SPATIAL_THRESHOLD_M,
    temporal_threshold_seconds=TEMPORAL_THRESHOLD_SECONDS
):
    """
    Find valid detection-level connections between two
    consecutive observed calendar dates.

    A pair is valid when:

        distance <= 1,000 m
        AND
        absolute time difference <= 12 hours

    Returns:
        DataFrame containing detection-level cross-day pairs.
    """

    if previous_day_df.empty or current_day_df.empty:
        return pd.DataFrame(
            columns=[
                "previous_index",
                "current_index",
                "distance_m",
                "time_difference_seconds"
            ]
        )

    previous = previous_day_df.copy()
    current = current_day_df.copy()

    previous = previous.reset_index(
        drop=False
    ).rename(
        columns={"index": "_previous_index"}
    )

    current = current.reset_index(
        drop=False
    ).rename(
        columns={"index": "_current_index"}
    )

    previous_xy = previous[
        ["x", "y"]
    ].to_numpy(dtype=np.float64)

    current_xy = current[
        ["x", "y"]
    ].to_numpy(dtype=np.float64)

    # --------------------------------------------------------
    # KD-tree on current-day observations
    # --------------------------------------------------------

    tree = cKDTree(current_xy)

    candidate_lists = tree.query_ball_point(
        previous_xy,
        r=spatial_threshold_m
    )

    rows = []

    previous_times = (
        previous["datetime"]
        .astype("int64")
        .to_numpy()
    )

    current_times = (
        current["datetime"]
        .astype("int64")
        .to_numpy()
    )

    # --------------------------------------------------------
    # Spatial candidates + temporal filtering
    # --------------------------------------------------------

    for i, candidates in enumerate(candidate_lists):

        if not candidates:
            continue

        candidates = np.asarray(
            candidates,
            dtype=np.int64
        )

        dt_seconds = np.abs(
            current_times[candidates]
            -
            previous_times[i]
        ) / 1_000_000_000.0

        valid_mask = (
            dt_seconds
            <= temporal_threshold_seconds
        )

        valid_candidates = candidates[
            valid_mask
        ]

        if len(valid_candidates) == 0:
            continue

        dx = (
            current_xy[valid_candidates, 0]
            -
            previous_xy[i, 0]
        )

        dy = (
            current_xy[valid_candidates, 1]
            -
            previous_xy[i, 1]
        )

        distances = np.sqrt(
            dx * dx + dy * dy
        )

        for j, distance_m in zip(
            valid_candidates,
            distances
        ):

            rows.append({
                "previous_index":
                    previous.loc[
                        i,
                        "_previous_index"
                    ],

                "current_index":
                    current.loc[
                        j,
                        "_current_index"
                    ],

                "distance_m":
                    float(distance_m),

                "time_difference_seconds":
                    float(
                        abs(
                            current_times[j]
                            -
                            previous_times[i]
                        )
                        /
                        1_000_000_000.0
                    )
            })

    return pd.DataFrame(rows)


In [37]:
# ============================================================
# MARCH 31 -> APRIL 1 CROSS-DAY REGRESSION
# ============================================================

DATE_A = pd.Timestamp(
    "2021-03-31"
).date()

DATE_B = pd.Timestamp(
    "2021-04-01"
).date()

day_a = df.loc[
    date_index[DATE_A]
].copy()

day_b = df.loc[
    date_index[DATE_B]
].copy()

print("=" * 70)
print("CROSS-DAY REGRESSION")
print("=" * 70)

print("Previous date:", DATE_A)
print("Current date :", DATE_B)

print(
    "Previous detections:",
    len(day_a)
)

print(
    "Current detections :",
    len(day_b)
)

cross_pairs = cross_day_detection_pairs(
    day_a,
    day_b
)

print(
    "\nDetection-level cross-day pairs:",
    len(cross_pairs)
)


CROSS-DAY REGRESSION
Previous date: 2021-03-31
Current date : 2021-04-01
Previous detections: 37801
Current detections : 36359

Detection-level cross-day pairs: 15887


In [38]:
# ============================================================
# CROSS-DAY EDGE CONSTRAINT QC
# ============================================================

print("=" * 70)
print("CROSS-DAY CONSTRAINT QC")
print("=" * 70)

if len(cross_pairs) > 0:

    print(
        "Maximum distance:",
        cross_pairs["distance_m"].max(),
        "m"
    )

    print(
        "Maximum time difference:",
        cross_pairs[
            "time_difference_seconds"
        ].max() / 3600.0,
        "hours"
    )

    print(
        "Minimum distance:",
        cross_pairs["distance_m"].min(),
        "m"
    )

    print(
        "Minimum time difference:",
        cross_pairs[
            "time_difference_seconds"
        ].min() / 3600.0,
        "hours"
    )

    assert (
        cross_pairs["distance_m"]
        <= SPATIAL_THRESHOLD_M
    ).all()

    assert (
        cross_pairs["time_difference_seconds"]
        <= TEMPORAL_THRESHOLD_SECONDS
    ).all()

print("\nExpected pairs: 15887")

assert len(cross_pairs) == 15887

print("✅ CROSS-DAY DETECTION REGRESSION PASSED")


CROSS-DAY CONSTRAINT QC
Maximum distance: 999.9921348562121 m
Maximum time difference: 12.0 hours
Minimum distance: 9.72777182086643 m
Minimum time difference: 9.55 hours

Expected pairs: 15887
✅ CROSS-DAY DETECTION REGRESSION PASSED


In [39]:
# ============================================================
# CREATE LOCAL CLUSTERS FOR CROSS-DAY REGRESSION
# ============================================================

clusters_a, labels_a = cluster_single_day(
    day_a,
    date_value=DATE_A
)

clusters_b, labels_b = cluster_single_day(
    day_b,
    date_value=DATE_B
)

print("=" * 70)
print("LOCAL CLUSTER REGRESSION")
print("=" * 70)

print("March 31 clusters:", len(clusters_a))
print("April 1 clusters  :", len(clusters_b))

assert len(clusters_a) == 12641
assert len(clusters_b) == 12901

print("✅ LOCAL CLUSTER BENCHMARKS PASSED")


TypeError: cluster_single_day() got an unexpected keyword argument 'date_value'

In [40]:
import inspect

print("SIGNATURE:")
print(inspect.signature(cluster_single_day))

print("\nSOURCE:")
print(inspect.getsource(cluster_single_day))


SIGNATURE:
(day_df, spatial_threshold_m=1000.0, temporal_threshold_seconds=43200.0)

SOURCE:
def cluster_single_day(
    day_df,
    spatial_threshold_m=SPATIAL_THRESHOLD_M,
    temporal_threshold_seconds=TEMPORAL_THRESHOLD_SECONDS
):
    """
    Daily fire-event clustering.

    Direct edge:
        distance <= 1,000 m
        AND
        |time difference| <= 12 h

    Final local clusters:
        connected components of the valid-edge graph.
    """

    if day_df.empty:
        return pd.DataFrame(), pd.DataFrame(), {
            "n_detections": 0,
            "n_spatial_pairs": 0,
            "n_valid_pairs": 0,
            "n_clusters": 0
        }

    work = day_df.copy()

    required = [
        "datetime",
        "x",
        "y",
        "frp",
        "latitude",
        "longitude",
        "satellite"
    ]

    missing = [
        c for c in required
        if c not in work.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns

In [41]:
# ============================================================
# MARCH 31 -> APRIL 1 LOCAL CLUSTER REGRESSION
# ============================================================

work_a, clusters_a, diagnostics_a = cluster_single_day(day_a)
work_b, clusters_b, diagnostics_b = cluster_single_day(day_b)

print("=" * 70)
print("LOCAL CLUSTER REGRESSION")
print("=" * 70)

print("March 31")
print("  Detections       :", diagnostics_a["n_detections"])
print("  Spatial pairs    :", diagnostics_a["n_spatial_pairs"])
print("  Valid pairs      :", diagnostics_a["n_valid_pairs"])
print("  Local clusters   :", diagnostics_a["n_clusters"])

print("\nApril 1")
print("  Detections       :", diagnostics_b["n_detections"])
print("  Spatial pairs    :", diagnostics_b["n_spatial_pairs"])
print("  Valid pairs      :", diagnostics_b["n_valid_pairs"])
print("  Local clusters   :", diagnostics_b["n_clusters"])

# ------------------------------------------------------------
# Exact validated benchmarks
# ------------------------------------------------------------

assert diagnostics_a["n_detections"] == 37801
assert diagnostics_a["n_spatial_pairs"] == 89322
assert diagnostics_a["n_valid_pairs"] == 75626
assert diagnostics_a["n_clusters"] == 12641

assert diagnostics_b["n_detections"] == 36359
assert diagnostics_b["n_spatial_pairs"] == 90202
assert diagnostics_b["n_valid_pairs"] == 74095
assert diagnostics_b["n_clusters"] == 12901

print("\n" + "=" * 70)
print("✅ MARCH 31 + APRIL 1 LOCAL CLUSTER REGRESSION PASSED")
print("=" * 70)


LOCAL CLUSTER REGRESSION
March 31
  Detections       : 37801
  Spatial pairs    : 89322
  Valid pairs      : 75626
  Local clusters   : 12641

April 1
  Detections       : 36359
  Spatial pairs    : 90202
  Valid pairs      : 74095
  Local clusters   : 12901

✅ MARCH 31 + APRIL 1 LOCAL CLUSTER REGRESSION PASSED


In [42]:
# ============================================================
# DETECTION -> LOCAL CLUSTER LINEAGE
# ============================================================

map_a = work_a[
    ["_original_index", "local_cluster_id"]
].copy()

map_b = work_b[
    ["_original_index", "local_cluster_id"]
].copy()

map_a = map_a.rename(
    columns={
        "_original_index": "detection_index"
    }
)

map_b = map_b.rename(
    columns={
        "_original_index": "detection_index"
    }
)

print("=" * 70)
print("DETECTION -> LOCAL CLUSTER LINEAGE QC")
print("=" * 70)

print("March 31 mappings:", len(map_a))
print("April 1 mappings  :", len(map_b))

print(
    "March 31 unique clusters:",
    map_a["local_cluster_id"].nunique()
)

print(
    "April 1 unique clusters:",
    map_b["local_cluster_id"].nunique()
)

print(
    "March 31 missing IDs:",
    map_a["local_cluster_id"].isna().sum()
)

print(
    "April 1 missing IDs:",
    map_b["local_cluster_id"].isna().sum()
)

assert len(map_a) == 37801
assert len(map_b) == 36359

assert map_a["local_cluster_id"].notna().all()
assert map_b["local_cluster_id"].notna().all()

assert map_a["local_cluster_id"].nunique() == 12641
assert map_b["local_cluster_id"].nunique() == 12901

print("\n✅ DETECTION-CLUSTER LINEAGE PASSED")


DETECTION -> LOCAL CLUSTER LINEAGE QC
March 31 mappings: 37801
April 1 mappings  : 36359
March 31 unique clusters: 12641
April 1 unique clusters: 12901
March 31 missing IDs: 0
April 1 missing IDs: 0

✅ DETECTION-CLUSTER LINEAGE PASSED


In [43]:
# ============================================================
# MAP DETECTION-LEVEL CROSS-DAY LINKS TO LOCAL CLUSTERS
# ============================================================

a_lookup = (
    map_a
    .set_index("detection_index")["local_cluster_id"]
)

b_lookup = (
    map_b
    .set_index("detection_index")["local_cluster_id"]
)

cluster_edges_raw = cross_pairs.copy()

cluster_edges_raw["cluster_a"] = (
    cluster_edges_raw["previous_index"]
    .map(a_lookup)
)

cluster_edges_raw["cluster_b"] = (
    cluster_edges_raw["current_index"]
    .map(b_lookup)
)

print("=" * 70)
print("CROSS-DAY CLUSTER RECONCILIATION")
print("=" * 70)

print(
    "Detection-level links:",
    len(cluster_edges_raw)
)

print(
    "Missing cluster A:",
    cluster_edges_raw["cluster_a"].isna().sum()
)

print(
    "Missing cluster B:",
    cluster_edges_raw["cluster_b"].isna().sum()
)

assert len(cluster_edges_raw) == 15887
assert cluster_edges_raw["cluster_a"].notna().all()
assert cluster_edges_raw["cluster_b"].notna().all()

# ------------------------------------------------------------
# Collapse detection links into unique cluster links
# ------------------------------------------------------------

cluster_edges = (
    cluster_edges_raw[
        ["cluster_a", "cluster_b"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "\nUnique cluster-level links:",
    len(cluster_edges)
)

assert len(cluster_edges) == 2066

print("\n✅ 15,887 → 2,066 RECONCILIATION PASSED")


CROSS-DAY CLUSTER RECONCILIATION
Detection-level links: 15887
Missing cluster A: 0
Missing cluster B: 0

Unique cluster-level links: 2066

✅ 15,887 → 2,066 RECONCILIATION PASSED


In [44]:
# ============================================================
# CROSS-DAY EDGE INTEGRITY QC
# ============================================================

print("=" * 70)
print("CROSS-DAY EDGE INTEGRITY QC")
print("=" * 70)

duplicate_edges = cluster_edges.duplicated(
    subset=["cluster_a", "cluster_b"]
).sum()

self_edges = (
    cluster_edges["cluster_a"]
    ==
    cluster_edges["cluster_b"]
).sum()

valid_a = set(
    clusters_a["local_cluster_id"]
)

valid_b = set(
    clusters_b["local_cluster_id"]
)

missing_a = sum(
    x not in valid_a
    for x in cluster_edges["cluster_a"]
)

missing_b = sum(
    x not in valid_b
    for x in cluster_edges["cluster_b"]
)

print("Unique cluster edges :", len(cluster_edges))
print("Duplicate edges      :", duplicate_edges)
print("Self edges           :", self_edges)
print("Missing A endpoints  :", missing_a)
print("Missing B endpoints  :", missing_b)

assert duplicate_edges == 0
assert self_edges == 0
assert missing_a == 0
assert missing_b == 0

print("\n✅ CROSS-DAY EDGE INTEGRITY PASSED")


CROSS-DAY EDGE INTEGRITY QC
Unique cluster edges : 2066
Duplicate edges      : 0
Self edges           : 0
Missing A endpoints  : 0
Missing B endpoints  : 0

✅ CROSS-DAY EDGE INTEGRITY PASSED


In [45]:
# ============================================================
# TWO-DAY GLOBAL UNION-FIND REGRESSION
# ============================================================

print("=" * 70)
print("TWO-DAY GLOBAL CONNECTED-COMPONENT REGRESSION")
print("=" * 70)

# ------------------------------------------------------------
# Combine all local cluster IDs
# ------------------------------------------------------------

all_cluster_ids = pd.concat(
    [
        clusters_a["local_cluster_id"],
        clusters_b["local_cluster_id"]
    ],
    ignore_index=True
)

print("Local cluster rows:", len(all_cluster_ids))
print(
    "Unique local cluster IDs:",
    all_cluster_ids.nunique()
)

assert len(all_cluster_ids) == 25542
assert all_cluster_ids.nunique() == 25542

# ------------------------------------------------------------
# Integer node mapping
# ------------------------------------------------------------

node_to_int = pd.Series(
    np.arange(
        len(all_cluster_ids),
        dtype=np.int64
    ),
    index=all_cluster_ids.to_numpy()
)

# ------------------------------------------------------------
# Union-Find
# ------------------------------------------------------------

n_nodes = len(all_cluster_ids)

parent = np.arange(
    n_nodes,
    dtype=np.int64
)

size = np.ones(
    n_nodes,
    dtype=np.int64
)


def find_root(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union_nodes(a, b):

    ra = find_root(a)
    rb = find_root(b)

    if ra == rb:
        return

    # Union by size
    if size[ra] < size[rb]:
        ra, rb = rb, ra

    parent[rb] = ra
    size[ra] += size[rb]


# ------------------------------------------------------------
# Apply cross-day cluster edges
# ------------------------------------------------------------

for row in cluster_edges.itertuples(index=False):

    a = node_to_int[row.cluster_a]
    b = node_to_int[row.cluster_b]

    union_nodes(
        int(a),
        int(b)
    )

# ------------------------------------------------------------
# Final roots
# ------------------------------------------------------------

roots = np.array(
    [
        find_root(i)
        for i in range(n_nodes)
    ],
    dtype=np.int64
)

n_components = np.unique(
    roots
).size

print("\n" + "=" * 70)
print("RESULT")
print("=" * 70)

print("Local clusters       :", n_nodes)
print("Cross-day edges      :", len(cluster_edges))
print("Global components    :", n_components)

# ------------------------------------------------------------
# Exact validated benchmark
# ------------------------------------------------------------

assert n_components == 23476

print("\n" + "=" * 70)
print("✅ TWO-DAY GLOBAL REGRESSION PASSED")
print("=" * 70)


TWO-DAY GLOBAL CONNECTED-COMPONENT REGRESSION
Local cluster rows: 25542
Unique local cluster IDs: 25542

RESULT
Local clusters       : 25542
Cross-day edges      : 2066
Global components    : 23476

✅ TWO-DAY GLOBAL REGRESSION PASSED


In [46]:
# ============================================================
# TWO-DAY GLOBAL EVENT STRUCTURE QC
# ============================================================

component_sizes = (
    pd.Series(roots)
    .value_counts()
)

print("=" * 70)
print("GLOBAL EVENT STRUCTURE")
print("=" * 70)

print("Global events:", len(component_sizes))

print("\nEvent-size distribution:")
print(component_sizes.describe())

print(
    "\nSingle-cluster events:",
    (component_sizes == 1).sum()
)

print(
    "Multi-cluster events:",
    (component_sizes > 1).sum()
)

print(
    "Maximum local clusters in one event:",
    component_sizes.max()
)

assert len(component_sizes) == 23476

print("\n✅ GLOBAL EVENT STRUCTURE QC PASSED")


GLOBAL EVENT STRUCTURE
Global events: 23476

Event-size distribution:
count    23476.000000
mean         1.088005
std          0.381253
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          9.000000
Name: count, dtype: float64

Single-cluster events: 21880
Multi-cluster events: 1596
Maximum local clusters in one event: 9

✅ GLOBAL EVENT STRUCTURE QC PASSED


In [47]:
from pathlib import Path
import pandas as pd
import numpy as np
import gc
import time
import json

# ============================================================
# FULL 2012–2026 LOCAL-CLUSTER PRODUCTION
# ============================================================

PRODUCTION_ROOT = Path(
    "/content/fire_event_production/full_2012_2026"
)

CLUSTER_DIR = PRODUCTION_ROOT / "clusters"
EDGE_DIR = PRODUCTION_ROOT / "edges"
EVENT_DIR = PRODUCTION_ROOT / "events"
LOG_DIR = PRODUCTION_ROOT / "logs"
MAPPING_DIR = PRODUCTION_ROOT / "mappings"

for directory in [
    PRODUCTION_ROOT,
    CLUSTER_DIR,
    EDGE_DIR,
    EVENT_DIR,
    LOG_DIR,
    MAPPING_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 70)
print("FULL 2012–2026 PRODUCTION DIRECTORIES")
print("=" * 70)

print("Production root:", PRODUCTION_ROOT)
print("Cluster dir    :", CLUSTER_DIR)
print("Edge dir       :", EDGE_DIR)
print("Event dir      :", EVENT_DIR)
print("Log dir        :", LOG_DIR)
print("Mapping dir    :", MAPPING_DIR)

print("\n✅ Directories ready")


FULL 2012–2026 PRODUCTION DIRECTORIES
Production root: /content/fire_event_production/full_2012_2026
Cluster dir    : /content/fire_event_production/full_2012_2026/clusters
Edge dir       : /content/fire_event_production/full_2012_2026/edges
Event dir      : /content/fire_event_production/full_2012_2026/events
Log dir        : /content/fire_event_production/full_2012_2026/logs
Mapping dir    : /content/fire_event_production/full_2012_2026/mappings

✅ Directories ready


In [48]:
# ============================================================
# OBSERVED DATE LIST
# ============================================================

observed_dates = sorted(
    date_index.keys()
)

print("=" * 70)
print("OBSERVED DATE RANGE")
print("=" * 70)

print("Observed dates:", len(observed_dates))
print("First date    :", observed_dates[0])
print("Last date     :", observed_dates[-1])

calendar_dates = pd.date_range(
    min(observed_dates),
    max(observed_dates),
    freq="D"
).date

missing_dates = sorted(
    set(calendar_dates) - set(observed_dates)
)

print("Calendar dates:", len(calendar_dates))
print("Missing dates :", len(missing_dates))

print("\nMissing dates:")
for d in missing_dates:
    print(" ", d)

assert len(observed_dates) == 5337
assert observed_dates[0] == pd.Timestamp("2012-01-20").date()
assert observed_dates[-1] == pd.Timestamp("2026-08-31").date()
assert missing_dates == [
    pd.Timestamp("2012-03-25").date()
]

print("\n✅ Date coverage validation passed")


OBSERVED DATE RANGE
Observed dates: 5337
First date    : 2012-01-20
Last date     : 2026-08-31
Calendar dates: 5338
Missing dates : 1

Missing dates:
  2012-03-25

✅ Date coverage validation passed


In [49]:
# ============================================================
# PRODUCTION LOG
# ============================================================

PRODUCTION_LOG = LOG_DIR / "local_cluster_production_log.csv"

LOG_COLUMNS = [
    "date",
    "status",
    "n_detections",
    "n_spatial_pairs",
    "n_valid_pairs",
    "n_clusters",
    "output_file",
    "elapsed_seconds",
    "error"
]

if PRODUCTION_LOG.exists():

    production_log = pd.read_csv(
        PRODUCTION_LOG
    )

else:

    production_log = pd.DataFrame(
        columns=LOG_COLUMNS
    )

print("=" * 70)
print("LOCAL-CLUSTER PRODUCTION LOG")
print("=" * 70)

print("Log file:", PRODUCTION_LOG)
print("Existing records:", len(production_log))

if len(production_log):
    print(
        "Completed dates:",
        (
            production_log["status"]
            .eq("COMPLETED")
            .sum()
        )
    )

print("✅ Production log ready")


LOCAL-CLUSTER PRODUCTION LOG
Log file: /content/fire_event_production/full_2012_2026/logs/local_cluster_production_log.csv
Existing records: 0
✅ Production log ready


In [50]:
# ============================================================
# LOG WRITER
# ============================================================

def save_production_log(log_df):
    """
    Save production progress atomically.
    """

    tmp_path = PRODUCTION_LOG.with_suffix(
        ".tmp.csv"
    )

    log_df.to_csv(
        tmp_path,
        index=False
    )

    tmp_path.replace(
        PRODUCTION_LOG
    )


In [51]:
# ============================================================
# FULL LOCAL-CLUSTER PRODUCTION RUNNER
# ============================================================

def run_full_local_cluster_production(
    observed_dates,
    date_index,
    df,
    cluster_dir,
    production_log,
    stop_after=None
):
    """
    Full daily local-cluster production.

    Each observed date is processed independently.

    Output:
        clusters_YYYY-MM-DD.parquet

    The missing calendar date 2012-03-25 is naturally skipped
    because it is absent from observed_dates.
    """

    print("=" * 70)
    print("STARTING FULL 2012–2026 LOCAL-CLUSTER PRODUCTION")
    print("=" * 70)

    total_dates = len(observed_dates)

    completed_dates = set(
        production_log.loc[
            production_log["status"] == "COMPLETED",
            "date"
        ].astype(str)
    )

    print("Total observed dates :", total_dates)
    print("Already completed    :", len(completed_dates))
    print("Remaining             :", total_dates - len(completed_dates))

    run_counter = 0

    for date_value in observed_dates:

        date_string = pd.Timestamp(
            date_value
        ).strftime("%Y-%m-%d")

        output_path = (
            cluster_dir /
            f"clusters_{date_string}.parquet"
        )

        # ----------------------------------------------------
        # Skip validated existing output
        # ----------------------------------------------------

        if (
            date_string in completed_dates
            and output_path.exists()
        ):

            continue

        # ----------------------------------------------------
        # Remove stale log entry for this date
        # ----------------------------------------------------

        production_log = production_log[
            production_log["date"].astype(str)
            != date_string
        ].copy()

        # ----------------------------------------------------
        # Extract daily detections
        # ----------------------------------------------------

        indices = date_index.get(
            pd.Timestamp(date_value).date()
        )

        if indices is None or len(indices) == 0:

            print(
                f"⚠️ No detections for {date_string}"
            )

            continue

        day_df = df.loc[
            indices
        ]

        start = time.time()

        print("\n" + "-" * 70)
        print(
            f"PROCESSING {date_string}"
        )
        print("-" * 70)

        print(
            "Detections:",
            len(day_df)
        )

        try:

            # ------------------------------------------------
            # VALIDATED DAILY CLUSTERING FUNCTION
            # ------------------------------------------------

            work_day, summary, diagnostics = (
                cluster_single_day(
                    day_df
                )
            )

            # ------------------------------------------------
            # Add date explicitly
            # ------------------------------------------------

            summary.insert(
                0,
                "date",
                date_string
            )

            # ------------------------------------------------
            # Write cluster summary
            # ------------------------------------------------

            summary.to_parquet(
                output_path,
                index=False
            )

            elapsed = time.time() - start

            # ------------------------------------------------
            # Record successful completion
            # ------------------------------------------------

            record = {
                "date": date_string,
                "status": "COMPLETED",
                "n_detections": diagnostics[
                    "n_detections"
                ],
                "n_spatial_pairs": diagnostics[
                    "n_spatial_pairs"
                ],
                "n_valid_pairs": diagnostics[
                    "n_valid_pairs"
                ],
                "n_clusters": diagnostics[
                    "n_clusters"
                ],
                "output_file": str(
                    output_path
                ),
                "elapsed_seconds": elapsed,
                "error": ""
            }

            production_log = pd.concat(
                [
                    production_log,
                    pd.DataFrame([record])
                ],
                ignore_index=True
            )

            save_production_log(
                production_log
            )

            print(
                "Spatial pairs:",
                diagnostics["n_spatial_pairs"]
            )

            print(
                "Valid pairs:",
                diagnostics["n_valid_pairs"]
            )

            print(
                "Local clusters:",
                diagnostics["n_clusters"]
            )

            print(
                f"Elapsed: {elapsed:.2f} s"
            )

            print(
                "✅ SAVED:",
                output_path.name
            )

            # ------------------------------------------------
            # Release daily objects
            # ------------------------------------------------

            del work_day
            del summary
            del day_df
            gc.collect()

            run_counter += 1

            # ------------------------------------------------
            # Optional controlled test
            # ------------------------------------------------

            if (
                stop_after is not None
                and run_counter >= stop_after
            ):

                print("\n")
                print("=" * 70)
                print(
                    "CONTROLLED STOP REACHED"
                )
                print("=" * 70)

                break

        except Exception as e:

            elapsed = time.time() - start

            record = {
                "date": date_string,
                "status": "FAILED",
                "n_detections": len(day_df),
                "n_spatial_pairs": np.nan,
                "n_valid_pairs": np.nan,
                "n_clusters": np.nan,
                "output_file": str(
                    output_path
                ),
                "elapsed_seconds": elapsed,
                "error": repr(e)
            }

            production_log = pd.concat(
                [
                    production_log,
                    pd.DataFrame([record])
                ],
                ignore_index=True
            )

            save_production_log(
                production_log
            )

            print(
                f"❌ FAILED: {date_string}"
            )

            print(
                "Error:",
                repr(e)
            )

            # Stop rather than silently continuing.
            raise

    print("\n")
    print("=" * 70)
    print("LOCAL-CLUSTER PRODUCTION RUN FINISHED")
    print("=" * 70)

    completed = (
        production_log["status"]
        .eq("COMPLETED")
        .sum()
    )

    failed = (
        production_log["status"]
        .eq("FAILED")
        .sum()
    )

    print("Completed:", completed)
    print("Failed   :", failed)

    return production_log


In [52]:
# ============================================================
# 10-DATE PRODUCTION SMOKE TEST
# ============================================================

test_dates = observed_dates[:10]

print("=" * 70)
print("10-DATE LOCAL-CLUSTER SMOKE TEST")
print("=" * 70)

for d in test_dates:
    print(d)

production_log = run_full_local_cluster_production(
    observed_dates=test_dates,
    date_index=date_index,
    df=df,
    cluster_dir=CLUSTER_DIR,
    production_log=production_log,
    stop_after=None
)

print("\n✅ 10-date smoke test completed")


10-DATE LOCAL-CLUSTER SMOKE TEST
2012-01-20
2012-01-21
2012-01-22
2012-01-23
2012-01-24
2012-01-25
2012-01-26
2012-01-27
2012-01-28
2012-01-29
STARTING FULL 2012–2026 LOCAL-CLUSTER PRODUCTION
Total observed dates : 10
Already completed    : 0
Remaining             : 10

----------------------------------------------------------------------
PROCESSING 2012-01-20
----------------------------------------------------------------------
Detections: 674


/tmp/ipykernel_3120/3215976647.py:167: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  production_log = pd.concat(


Spatial pairs: 395
Valid pairs: 343
Local clusters: 459
Elapsed: 1.66 s
✅ SAVED: clusters_2012-01-20.parquet

----------------------------------------------------------------------
PROCESSING 2012-01-21
----------------------------------------------------------------------
Detections: 719
Spatial pairs: 468
Valid pairs: 404
Local clusters: 459
Elapsed: 1.01 s
✅ SAVED: clusters_2012-01-21.parquet

----------------------------------------------------------------------
PROCESSING 2012-01-22
----------------------------------------------------------------------
Detections: 693
Spatial pairs: 563
Valid pairs: 507
Local clusters: 446
Elapsed: 0.81 s
✅ SAVED: clusters_2012-01-22.parquet

----------------------------------------------------------------------
PROCESSING 2012-01-23
----------------------------------------------------------------------
Detections: 1013
Spatial pairs: 699
Valid pairs: 633
Local clusters: 694
Elapsed: 1.47 s
✅ SAVED: clusters_2012-01-23.parquet

-------------------

In [53]:
# ============================================================
# SMOKE TEST OUTPUT QC
# ============================================================

smoke_files = sorted(
    CLUSTER_DIR.glob("clusters_*.parquet")
)

print("=" * 70)
print("SMOKE TEST OUTPUT QC")
print("=" * 70)

print("Cluster files:", len(smoke_files))

for p in smoke_files:
    print(
        p.name,
        f"{p.stat().st_size / 1024**2:.3f} MB"
    )

assert len(smoke_files) == 10

print("\n✅ 10 cluster files created")


SMOKE TEST OUTPUT QC
Cluster files: 10
clusters_2012-01-20.parquet 0.039 MB
clusters_2012-01-21.parquet 0.040 MB
clusters_2012-01-22.parquet 0.038 MB
clusters_2012-01-23.parquet 0.052 MB
clusters_2012-01-24.parquet 0.052 MB
clusters_2012-01-25.parquet 0.046 MB
clusters_2012-01-26.parquet 0.041 MB
clusters_2012-01-27.parquet 0.049 MB
clusters_2012-01-28.parquet 0.048 MB
clusters_2012-01-29.parquet 0.052 MB

✅ 10 cluster files created


In [54]:
# ============================================================
# FULL 2012–2026 LOCAL-CLUSTER PRODUCTION
# ============================================================

production_log = run_full_local_cluster_production(
    observed_dates=observed_dates,
    date_index=date_index,
    df=df,
    cluster_dir=CLUSTER_DIR,
    production_log=production_log
)

print("\n")
print("=" * 70)
print("FULL LOCAL-CLUSTER PRODUCTION COMPLETE")
print("=" * 70)

print(
    "Completed dates:",
    (
        production_log["status"]
        == "COMPLETED"
    ).sum()
)

print(
    "Failed dates:",
    (
        production_log["status"]
        == "FAILED"
    ).sum()
)


Streaming output truncated to the last 5000 lines.
Detections: 11663
Spatial pairs: 34739
Valid pairs: 31757
Local clusters: 3789
Elapsed: 5.25 s
✅ SAVED: clusters_2025-04-20.parquet

----------------------------------------------------------------------
PROCESSING 2025-04-21
----------------------------------------------------------------------
Detections: 13213
Spatial pairs: 36663
Valid pairs: 34284
Local clusters: 4307
Elapsed: 4.14 s
✅ SAVED: clusters_2025-04-21.parquet

----------------------------------------------------------------------
PROCESSING 2025-04-22
----------------------------------------------------------------------
Detections: 20011
Spatial pairs: 64378
Valid pairs: 58843
Local clusters: 5611
Elapsed: 6.13 s
✅ SAVED: clusters_2025-04-22.parquet

----------------------------------------------------------------------
PROCESSING 2025-04-23
----------------------------------------------------------------------
Detections: 24283
Spatial pairs: 85048
Valid pairs: 76947


In [55]:
# ============================================================
# FULL 2012–2026 EVENT PRODUCTION — SETUP
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import json
import gc

# ------------------------------------------------------------
# DIRECTORIES
# ------------------------------------------------------------

PRODUCTION_ROOT = Path(
    "/content/fire_event_production/full_2012_2026"
)

CLUSTER_DIR = PRODUCTION_ROOT / "clusters"
EDGE_DIR = PRODUCTION_ROOT / "edges"
EVENT_DIR = PRODUCTION_ROOT / "events"
LOG_DIR = PRODUCTION_ROOT / "logs"
MAPPING_DIR = PRODUCTION_ROOT / "mappings"

for d in [
    PRODUCTION_ROOT,
    CLUSTER_DIR,
    EDGE_DIR,
    EVENT_DIR,
    LOG_DIR,
    MAPPING_DIR
]:
    d.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 70)
print("EVENT PRODUCTION DIRECTORIES")
print("=" * 70)

print("Production root:", PRODUCTION_ROOT)
print("Cluster dir    :", CLUSTER_DIR)
print("Edge dir       :", EDGE_DIR)
print("Event dir      :", EVENT_DIR)
print("Log dir        :", LOG_DIR)
print("Mapping dir    :", MAPPING_DIR)

print("\n✅ Directories ready")


EVENT PRODUCTION DIRECTORIES
Production root: /content/fire_event_production/full_2012_2026
Cluster dir    : /content/fire_event_production/full_2012_2026/clusters
Edge dir       : /content/fire_event_production/full_2012_2026/edges
Event dir      : /content/fire_event_production/full_2012_2026/events
Log dir        : /content/fire_event_production/full_2012_2026/logs
Mapping dir    : /content/fire_event_production/full_2012_2026/mappings

✅ Directories ready


In [56]:
# ============================================================
# LOCAL-CLUSTER PRODUCTION QC
# ============================================================

cluster_files = sorted(
    CLUSTER_DIR.glob("clusters_*.parquet")
)

print("=" * 70)
print("LOCAL-CLUSTER PRODUCTION QC")
print("=" * 70)

print("Cluster files:", len(cluster_files))

assert len(cluster_files) == len(observed_dates), (
    f"Expected {len(observed_dates)} cluster files, "
    f"found {len(cluster_files)}"
)

print("Observed dates:", len(observed_dates))

print(
    "Expected files:",
    len(observed_dates)
)

print("\nFirst 5:")
for p in cluster_files[:5]:
    print(" ", p.name)

print("\nLast 5:")
for p in cluster_files[-5:]:
    print(" ", p.name)

print("\n✅ All observed dates have cluster files")


LOCAL-CLUSTER PRODUCTION QC
Cluster files: 5337
Observed dates: 5337
Expected files: 5337

First 5:
  clusters_2012-01-20.parquet
  clusters_2012-01-21.parquet
  clusters_2012-01-22.parquet
  clusters_2012-01-23.parquet
  clusters_2012-01-24.parquet

Last 5:
  clusters_2026-08-27.parquet
  clusters_2026-08-28.parquet
  clusters_2026-08-29.parquet
  clusters_2026-08-30.parquet
  clusters_2026-08-31.parquet

✅ All observed dates have cluster files


In [57]:
# ============================================================
# CLUSTER FILE SCHEMA QC
# ============================================================

required_cluster_columns = [
    "local_cluster_id",
    "start_time",
    "end_time",
    "n_detections",
    "centroid_x",
    "centroid_y",
    "total_frp",
    "mean_frp",
    "max_frp",
    "n_observation_times",
    "duration_hours",
    "satellites",
    "n_satellites",
    "max_radius_m",
    "max_radius_km",
    "centroid_lon",
    "centroid_lat"
]

sample_file = cluster_files[0]

sample_cluster = pd.read_parquet(
    sample_file
)

print("=" * 70)
print("CLUSTER SCHEMA QC")
print("=" * 70)

print("Sample file:", sample_file.name)

print("\nColumns:")
for c in sample_cluster.columns:
    print(" ", c)

missing = [
    c
    for c in required_cluster_columns
    if c not in sample_cluster.columns
]

print("\nMissing required columns:", missing)

assert not missing

print("\nRows in sample:", len(sample_cluster))

assert sample_cluster["local_cluster_id"].notna().all()
assert sample_cluster["centroid_lon"].notna().all()
assert sample_cluster["centroid_lat"].notna().all()

print("\n✅ Cluster schema QC passed")


CLUSTER SCHEMA QC
Sample file: clusters_2012-01-20.parquet

Columns:
  date
  local_cluster_id
  start_time
  end_time
  n_detections
  centroid_x
  centroid_y
  total_frp
  mean_frp
  max_frp
  n_observation_times
  duration_hours
  satellites
  n_satellites
  max_radius_m
  max_radius_km
  centroid_lon
  centroid_lat

Missing required columns: []

Rows in sample: 459

✅ Cluster schema QC passed


In [58]:
# ============================================================
# BUILD LOCAL-CLUSTER INVENTORY
# ============================================================

inventory_rows = []

for p in cluster_files:

    date_text = p.stem.replace(
        "clusters_",
        ""
    )

    try:
        date_value = pd.Timestamp(
            date_text
        ).date()
    except Exception:
        print("⚠️ Could not parse:", p.name)
        continue

    inventory_rows.append({
        "date": date_value,
        "cluster_file": str(p),
        "file_size_mb": (
            p.stat().st_size / 1024**2
        )
    })

cluster_inventory = pd.DataFrame(
    inventory_rows
).sort_values(
    "date"
).reset_index(drop=True)

print("=" * 70)
print("LOCAL-CLUSTER INVENTORY")
print("=" * 70)

print(
    "Inventory rows:",
    len(cluster_inventory)
)

print(
    "First date:",
    cluster_inventory["date"].min()
)

print(
    "Last date:",
    cluster_inventory["date"].max()
)

print(
    "Total cluster-file size MB:",
    cluster_inventory["file_size_mb"].sum()
)

assert len(cluster_inventory) == len(observed_dates)

print("\n✅ Cluster inventory ready")


LOCAL-CLUSTER INVENTORY
Inventory rows: 5337
First date: 2012-01-20
Last date: 2026-08-31
Total cluster-file size MB: 531.9904594421387

✅ Cluster inventory ready


In [59]:
# ============================================================
# BUILD MASTER LOCAL-CLUSTER TABLE
# ============================================================

local_cluster_parts = []

for i, row in cluster_inventory.iterrows():

    p = Path(
        row["cluster_file"]
    )

    part = pd.read_parquet(
        p
    )

    part["date"] = pd.Timestamp(
        row["date"]
    ).date()

    local_cluster_parts.append(
        part
    )

    if (i + 1) % 250 == 0:
        print(
            f"Loaded {i + 1:,} / "
            f"{len(cluster_inventory):,} cluster files"
        )

local_clusters = pd.concat(
    local_cluster_parts,
    ignore_index=True
)

del local_cluster_parts
gc.collect()

print("=" * 70)
print("MASTER LOCAL-CLUSTER TABLE")
print("=" * 70)

print(
    "Rows:",
    len(local_clusters)
)

print(
    "Unique local clusters:",
    local_clusters["local_cluster_id"].nunique()
)

print(
    "Date range:",
    local_clusters["date"].min(),
    "→",
    local_clusters["date"].max()
)

assert (
    local_clusters["local_cluster_id"].nunique()
    == len(local_clusters)
)

print("\n✅ Master local-cluster table created")


Loaded 250 / 5,337 cluster files
Loaded 500 / 5,337 cluster files
Loaded 750 / 5,337 cluster files
Loaded 1,000 / 5,337 cluster files
Loaded 1,250 / 5,337 cluster files
Loaded 1,500 / 5,337 cluster files
Loaded 1,750 / 5,337 cluster files
Loaded 2,000 / 5,337 cluster files
Loaded 2,250 / 5,337 cluster files
Loaded 2,500 / 5,337 cluster files
Loaded 2,750 / 5,337 cluster files
Loaded 3,000 / 5,337 cluster files
Loaded 3,250 / 5,337 cluster files
Loaded 3,500 / 5,337 cluster files
Loaded 3,750 / 5,337 cluster files
Loaded 4,000 / 5,337 cluster files
Loaded 4,250 / 5,337 cluster files
Loaded 4,500 / 5,337 cluster files
Loaded 4,750 / 5,337 cluster files
Loaded 5,000 / 5,337 cluster files
Loaded 5,250 / 5,337 cluster files
MASTER LOCAL-CLUSTER TABLE
Rows: 7589305
Unique local clusters: 7589305
Date range: 2012-01-20 → 2026-08-31

✅ Master local-cluster table created


In [60]:
# ============================================================
# BUILD CONSECUTIVE OBSERVED-DATE PAIRS
# ============================================================

observed_dates_sorted = sorted(
    pd.Timestamp(d).date()
    for d in observed_dates
)

date_pairs = []

for previous_date, current_date in zip(
    observed_dates_sorted[:-1],
    observed_dates_sorted[1:]
):

    calendar_gap = (
        pd.Timestamp(current_date)
        - pd.Timestamp(previous_date)
    ).days

    # Only connect genuinely consecutive calendar dates
    if calendar_gap == 1:

        date_pairs.append(
            (
                previous_date,
                current_date
            )
        )

print("=" * 70)
print("CROSS-DAY DATE-PAIR SCHEDULE")
print("=" * 70)

print(
    "Observed dates:",
    len(observed_dates_sorted)
)

print(
    "Valid consecutive date pairs:",
    len(date_pairs)
)

print("\nFirst 5:")
for pair in date_pairs[:5]:
    print(pair)

print("\nLast 5:")
for pair in date_pairs[-5:]:
    print(pair)

print("\nExpected missing-date break:")
print(
    any(
        a == pd.Timestamp("2012-03-24").date()
        and b == pd.Timestamp("2012-03-26").date()
        for a, b in date_pairs
    )
)

assert not any(
    a == pd.Timestamp("2012-03-24").date()
    and b == pd.Timestamp("2012-03-26").date()
    for a, b in date_pairs
)

print("\n✅ Date-pair schedule passed")


CROSS-DAY DATE-PAIR SCHEDULE
Observed dates: 5337
Valid consecutive date pairs: 5335

First 5:
(datetime.date(2012, 1, 20), datetime.date(2012, 1, 21))
(datetime.date(2012, 1, 21), datetime.date(2012, 1, 22))
(datetime.date(2012, 1, 22), datetime.date(2012, 1, 23))
(datetime.date(2012, 1, 23), datetime.date(2012, 1, 24))
(datetime.date(2012, 1, 24), datetime.date(2012, 1, 25))

Last 5:
(datetime.date(2026, 8, 26), datetime.date(2026, 8, 27))
(datetime.date(2026, 8, 27), datetime.date(2026, 8, 28))
(datetime.date(2026, 8, 28), datetime.date(2026, 8, 29))
(datetime.date(2026, 8, 29), datetime.date(2026, 8, 30))
(datetime.date(2026, 8, 30), datetime.date(2026, 8, 31))

Expected missing-date break:
False

✅ Date-pair schedule passed


In [61]:
# ============================================================
# FIND CROSS-DAY FUNCTIONS
# ============================================================

import inspect

keywords = [
    "cross",
    "day",
    "temporal",
    "edge",
    "boundary",
    "cluster"
]

print("=" * 70)
print("AVAILABLE CROSS-DAY FUNCTIONS")
print("=" * 70)

for name, obj in globals().items():

    if inspect.isfunction(obj):

        if any(
            k in name.lower()
            for k in keywords
        ):

            try:
                sig = inspect.signature(obj)
            except Exception:
                sig = "(signature unavailable)"

            print(
                f"{name}{sig}"
            )



AVAILABLE CROSS-DAY FUNCTIONS


RuntimeError: dictionary changed size during iteration

In [62]:
import inspect

for name, obj in list(globals().items()):
    if inspect.isfunction(obj):
        print(name)

production_log_path
count_files
cluster_single_day
cross_day_detection_pairs
find_root
union_nodes
save_production_log
run_full_local_cluster_production


In [63]:
# ============================================================
# INSPECT EXISTING CROSS-DAY / UNION-FIND FUNCTIONS
# ============================================================

import inspect

functions_to_inspect = [
    "cross_day_detection_pairs",
    "find_root",
    "union_nodes",
    "save_production_log",
    "count_files",
]

print("=" * 70)
print("EXISTING CROSS-DAY / EVENT FUNCTIONS")
print("=" * 70)

for name in functions_to_inspect:

    obj = globals().get(name)

    print("\n" + "-" * 70)
    print(name)
    print("-" * 70)

    if obj is None:
        print("❌ NOT FOUND")
        continue

    try:
        print("SIGNATURE:")
        print(inspect.signature(obj))
    except Exception as e:
        print("Signature unavailable:", e)

    try:
        print("\nSOURCE:")
        print(inspect.getsource(obj))
    except Exception as e:
        print("Source unavailable:", e)


EXISTING CROSS-DAY / EVENT FUNCTIONS

----------------------------------------------------------------------
cross_day_detection_pairs
----------------------------------------------------------------------
SIGNATURE:
(previous_day_df, current_day_df, spatial_threshold_m=1000.0, temporal_threshold_seconds=43200.0)

SOURCE:
def cross_day_detection_pairs(
    previous_day_df,
    current_day_df,
    spatial_threshold_m=SPATIAL_THRESHOLD_M,
    temporal_threshold_seconds=TEMPORAL_THRESHOLD_SECONDS
):
    """
    Find valid detection-level connections between two
    consecutive observed calendar dates.

    A pair is valid when:

        distance <= 1,000 m
        AND
        absolute time difference <= 12 hours

    Returns:
        DataFrame containing detection-level cross-day pairs.
    """

    if previous_day_df.empty or current_day_df.empty:
        return pd.DataFrame(
            columns=[
                "previous_index",
                "current_index",
                "distanc

In [64]:
# ============================================================
# CHECK CROSS-DAY ENVIRONMENT
# ============================================================

required_names = [
    "CLUSTER_DIR",
    "EDGE_DIR",
    "EVENT_DIR",
    "MAPPING_DIR",
    "LOG_DIR",
    "PRODUCTION_ROOT",
    "observed_dates",
    "date_index",
    "df",
    "SPATIAL_THRESHOLD_M",
    "TEMPORAL_THRESHOLD_SECONDS",
]

print("=" * 70)
print("CROSS-DAY ENVIRONMENT CHECK")
print("=" * 70)

for name in required_names:

    exists = name in globals()

    print(
        f"{name:<32} : "
        f"{'AVAILABLE' if exists else 'MISSING'}"
    )


CROSS-DAY ENVIRONMENT CHECK
CLUSTER_DIR                      : AVAILABLE
EDGE_DIR                         : AVAILABLE
EVENT_DIR                        : AVAILABLE
MAPPING_DIR                      : AVAILABLE
LOG_DIR                          : AVAILABLE
PRODUCTION_ROOT                  : AVAILABLE
observed_dates                   : AVAILABLE
date_index                       : AVAILABLE
df                               : AVAILABLE
SPATIAL_THRESHOLD_M              : AVAILABLE
TEMPORAL_THRESHOLD_SECONDS       : AVAILABLE


In [65]:
# ============================================================
# CROSS-DAY PRODUCTION DIRECTORY CHECK
# ============================================================

from pathlib import Path

print("=" * 70)
print("CROSS-DAY PRODUCTION DIRECTORIES")
print("=" * 70)

print("PRODUCTION_ROOT:", PRODUCTION_ROOT)
print("CLUSTER_DIR    :", CLUSTER_DIR)
print("EDGE_DIR       :", EDGE_DIR)
print("EVENT_DIR      :", EVENT_DIR)
print("MAPPING_DIR    :", MAPPING_DIR)
print("LOG_DIR        :", LOG_DIR)

for name, directory in [
    ("PRODUCTION_ROOT", PRODUCTION_ROOT),
    ("CLUSTER_DIR", CLUSTER_DIR),
    ("EDGE_DIR", EDGE_DIR),
    ("EVENT_DIR", EVENT_DIR),
    ("MAPPING_DIR", MAPPING_DIR),
    ("LOG_DIR", LOG_DIR),
]:

    directory = Path(directory)
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

    print(
        f"{name:<16}: "
        f"{directory.exists()} "
        f"{directory}"
    )

print("\n✅ Cross-day directories ready")


CROSS-DAY PRODUCTION DIRECTORIES
PRODUCTION_ROOT: /content/fire_event_production/full_2012_2026
CLUSTER_DIR    : /content/fire_event_production/full_2012_2026/clusters
EDGE_DIR       : /content/fire_event_production/full_2012_2026/edges
EVENT_DIR      : /content/fire_event_production/full_2012_2026/events
MAPPING_DIR    : /content/fire_event_production/full_2012_2026/mappings
LOG_DIR        : /content/fire_event_production/full_2012_2026/logs
PRODUCTION_ROOT : True /content/fire_event_production/full_2012_2026
CLUSTER_DIR     : True /content/fire_event_production/full_2012_2026/clusters
EDGE_DIR        : True /content/fire_event_production/full_2012_2026/edges
EVENT_DIR       : True /content/fire_event_production/full_2012_2026/events
MAPPING_DIR     : True /content/fire_event_production/full_2012_2026/mappings
LOG_DIR         : True /content/fire_event_production/full_2012_2026/logs

✅ Cross-day directories ready


In [66]:
# ============================================================
# FINAL LOCAL-CLUSTER INPUT QC
# ============================================================

cluster_files = sorted(
    Path(CLUSTER_DIR).glob(
        "clusters_*.parquet"
    )
)

print("=" * 70)
print("LOCAL-CLUSTER INPUT QC")
print("=" * 70)

print(
    "Cluster files:",
    len(cluster_files)
)

print(
    "Observed dates:",
    len(observed_dates)
)

assert len(cluster_files) == len(observed_dates)

print("\nFirst:")
print(cluster_files[0])

print("\nLast:")
print(cluster_files[-1])

print(
    "\nTotal cluster files:",
    len(cluster_files)
)

print("✅ Local-cluster input QC passed")


LOCAL-CLUSTER INPUT QC
Cluster files: 5337
Observed dates: 5337

First:
/content/fire_event_production/full_2012_2026/clusters/clusters_2012-01-20.parquet

Last:
/content/fire_event_production/full_2012_2026/clusters/clusters_2026-08-31.parquet

Total cluster files: 5337
✅ Local-cluster input QC passed


In [67]:
# ============================================================
# CHECK EXISTING DETECTION → LOCAL-CLUSTER MAPPINGS
# ============================================================

mapping_files = sorted(
    Path(MAPPING_DIR).glob("*")
)

print("=" * 70)
print("EXISTING MAPPING FILES")
print("=" * 70)

print(
    "Files:",
    len(mapping_files)
)

for p in mapping_files[:20]:
    print(p.name)


EXISTING MAPPING FILES
Files: 0


In [68]:
mapping_files = sorted(
    Path(MAPPING_DIR).glob("*")
)

print("=" * 70)
print("EXISTING MAPPING FILES")
print("=" * 70)

print("Files:", len(mapping_files))

for p in mapping_files[:20]:
    print(p.name)


EXISTING MAPPING FILES
Files: 0


In [69]:
# ============================================================
# CROSS-DAY EDGE PRODUCTION — INITIALIZATION + QC
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time

print("=" * 70)
print("CROSS-DAY EDGE PRODUCTION")
print("=" * 70)

# ------------------------------------------------------------
# DIRECTORIES
# ------------------------------------------------------------

PRODUCTION_ROOT = Path(
    "/content/fire_event_production/full_2012_2026"
)

CLUSTER_DIR = PRODUCTION_ROOT / "clusters"
EDGE_DIR = PRODUCTION_ROOT / "edges"
EVENT_DIR = PRODUCTION_ROOT / "events"
MAPPING_DIR = PRODUCTION_ROOT / "mappings"
LOG_DIR = PRODUCTION_ROOT / "logs"

for directory in [
    PRODUCTION_ROOT,
    CLUSTER_DIR,
    EDGE_DIR,
    EVENT_DIR,
    MAPPING_DIR,
    LOG_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# INPUT INVENTORY
# ------------------------------------------------------------

cluster_files = sorted(
    CLUSTER_DIR.glob("clusters_*.parquet")
)

print("Cluster files:", len(cluster_files))

assert len(cluster_files) == len(observed_dates), (
    f"Expected {len(observed_dates)} cluster files, "
    f"found {len(cluster_files)}"
)

# ------------------------------------------------------------
# DATE PAIRS
# ------------------------------------------------------------

observed_date_values = [
    pd.Timestamp(d).date()
    for d in observed_dates
]

date_pairs = []

for previous_date, current_date in zip(
    observed_date_values[:-1],
    observed_date_values[1:]
):

    # Only consecutive calendar dates are paired.
    if (
        current_date
        - previous_date
    ).days == 1:

        date_pairs.append(
            (
                previous_date,
                current_date
            )
        )

print("Observed dates:", len(observed_date_values))
print("Cross-day date pairs:", len(date_pairs))

# ------------------------------------------------------------
# EXPECTED COUNT
# ------------------------------------------------------------

expected_pairs = sum(
    1
    for a, b in zip(
        observed_date_values[:-1],
        observed_date_values[1:]
    )
    if (b - a).days == 1
)

assert len(date_pairs) == expected_pairs

print(
    "Expected consecutive pairs:",
    expected_pairs
)

# ------------------------------------------------------------
# EXISTING OUTPUTS
# ------------------------------------------------------------

existing_edge_files = sorted(
    EDGE_DIR.glob("*.parquet")
)

print(
    "Existing edge files:",
    len(existing_edge_files)
)

print("\nFirst 5 date pairs:")

for pair in date_pairs[:5]:
    print(pair)

print("\nLast 5 date pairs:")

for pair in date_pairs[-5:]:
    print(pair)

print("\n" + "=" * 70)
print("READY FOR CROSS-DAY EDGE PRODUCTION")
print("=" * 70)


CROSS-DAY EDGE PRODUCTION
Cluster files: 5337
Observed dates: 5337
Cross-day date pairs: 5335
Expected consecutive pairs: 5335
Existing edge files: 0

First 5 date pairs:
(datetime.date(2012, 1, 20), datetime.date(2012, 1, 21))
(datetime.date(2012, 1, 21), datetime.date(2012, 1, 22))
(datetime.date(2012, 1, 22), datetime.date(2012, 1, 23))
(datetime.date(2012, 1, 23), datetime.date(2012, 1, 24))
(datetime.date(2012, 1, 24), datetime.date(2012, 1, 25))

Last 5 date pairs:
(datetime.date(2026, 8, 26), datetime.date(2026, 8, 27))
(datetime.date(2026, 8, 27), datetime.date(2026, 8, 28))
(datetime.date(2026, 8, 28), datetime.date(2026, 8, 29))
(datetime.date(2026, 8, 29), datetime.date(2026, 8, 30))
(datetime.date(2026, 8, 30), datetime.date(2026, 8, 31))

READY FOR CROSS-DAY EDGE PRODUCTION


In [70]:
# ============================================================
# SINGLE CROSS-DAY PRODUCTION TEST
# ============================================================

DATE_A = pd.Timestamp("2012-01-20").date()
DATE_B = pd.Timestamp("2012-01-21").date()

print("=" * 70)
print("SINGLE CROSS-DAY EDGE TEST")
print("=" * 70)

print("Previous date:", DATE_A)
print("Current date :", DATE_B)

# ------------------------------------------------------------
# LOAD DETECTIONS
# ------------------------------------------------------------

day_a = df.loc[
    date_index[DATE_A]
].copy()

day_b = df.loc[
    date_index[DATE_B]
].copy()

print("Previous detections:", len(day_a))
print("Current detections :", len(day_b))

# ------------------------------------------------------------
# DETECTION-LEVEL CROSS-DAY MATCHING
# ------------------------------------------------------------

t0 = time.time()

cross_pairs_test = cross_day_detection_pairs(
    day_a,
    day_b
)

elapsed = time.time() - t0

print(
    "Detection-level cross-day pairs:",
    len(cross_pairs_test)
)

print(
    "Elapsed:",
    round(elapsed, 2),
    "seconds"
)

# ------------------------------------------------------------
# BASIC QC
# ------------------------------------------------------------

required_columns = [
    "previous_index",
    "current_index",
    "distance_m",
    "time_difference_seconds"
]

missing = [
    c
    for c in required_columns
    if c not in cross_pairs_test.columns
]

assert not missing, (
    f"Missing columns: {missing}"
)

assert (
    cross_pairs_test["distance_m"]
    <= SPATIAL_THRESHOLD_M
).all()

assert (
    cross_pairs_test["time_difference_seconds"]
    <= TEMPORAL_THRESHOLD_SECONDS
).all()

print("\nDistance maximum:",
      cross_pairs_test["distance_m"].max())

print(
    "Time difference maximum:",
    cross_pairs_test["time_difference_seconds"].max()
)

print("\n✅ SINGLE CROSS-DAY DETECTION TEST PASSED")


SINGLE CROSS-DAY EDGE TEST
Previous date: 2012-01-20
Current date : 2012-01-21
Previous detections: 674
Current detections : 719
Detection-level cross-day pairs: 65
Elapsed: 0.02 seconds

Distance maximum: 970.4125528799783
Time difference maximum: 40440.0

✅ SINGLE CROSS-DAY DETECTION TEST PASSED


In [71]:
# ============================================================
# MAP CROSS-DAY DETECTIONS → LOCAL CLUSTERS
# ============================================================

# ------------------------------------------------------------
# LOCAL CLUSTER MAPS
# ------------------------------------------------------------

map_a = (
    df.loc[
        date_index[DATE_A]
    ]
    .reset_index()
    [
        ["index"]
    ]
)

map_a = map_a.rename(
    columns={
        "index": "detection_index"
    }
)

# We need the cluster assignment for each detection.
# Reconstruct it from the corresponding daily cluster run.


In [72]:
# ============================================================
# DETECTION → LOCAL-CLUSTER MAPPING TEST
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

MAPPING_DIR = Path(
    "/content/fire_event_production/full_2012_2026/mappings"
)

MAPPING_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TEST_DATE = pd.Timestamp("2012-01-20").date()

print("=" * 70)
print("DETECTION → LOCAL-CLUSTER MAPPING TEST")
print("=" * 70)

print("Date:", TEST_DATE)

# ------------------------------------------------------------
# LOAD ORIGINAL DETECTIONS
# ------------------------------------------------------------

day_df = df.loc[
    date_index[TEST_DATE]
].copy()

print(
    "Detections:",
    len(day_df)
)

# ------------------------------------------------------------
# RECREATE THE ALREADY-VALIDATED DAILY CLUSTERING
# ------------------------------------------------------------

work_test, summary_test, diagnostics_test = (
    cluster_single_day(
        day_df
    )
)

print(
    "Local clusters:",
    diagnostics_test["n_clusters"]
)

# ------------------------------------------------------------
# EXTRACT ONLY DETECTION → CLUSTER LINEAGE
# ------------------------------------------------------------

mapping_test = work_test[
    [
        "_original_index",
        "local_cluster_id"
    ]
].copy()

mapping_test = mapping_test.rename(
    columns={
        "_original_index": "detection_index"
    }
)

# ------------------------------------------------------------
# QC
# ------------------------------------------------------------

print("\nMapping rows:", len(mapping_test))

print(
    "Unique detections:",
    mapping_test["detection_index"].nunique()
)

print(
    "Unique clusters:",
    mapping_test["local_cluster_id"].nunique()
)

print(
    "Missing cluster IDs:",
    mapping_test["local_cluster_id"].isna().sum()
)

assert len(mapping_test) == len(day_df)

assert (
    mapping_test["detection_index"].nunique()
    == len(day_df)
)

assert (
    mapping_test["local_cluster_id"]
    .notna()
    .all()
)

assert (
    mapping_test["local_cluster_id"]
    .nunique()
    == diagnostics_test["n_clusters"]
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

mapping_path = (
    MAPPING_DIR
    / f"detection_cluster_mapping_{TEST_DATE}.parquet"
)

mapping_test.to_parquet(
    mapping_path,
    index=False
)

print(
    "\nSaved:",
    mapping_path
)

print("\n" + "=" * 70)
print("✅ DETECTION → LOCAL-CLUSTER MAPPING TEST PASSED")
print("=" * 70)


DETECTION → LOCAL-CLUSTER MAPPING TEST
Date: 2012-01-20
Detections: 674
Local clusters: 459

Mapping rows: 674
Unique detections: 674
Unique clusters: 459
Missing cluster IDs: 0

Saved: /content/fire_event_production/full_2012_2026/mappings/detection_cluster_mapping_2012-01-20.parquet

✅ DETECTION → LOCAL-CLUSTER MAPPING TEST PASSED


In [73]:
# ============================================================
# FULL DETECTION → LOCAL-CLUSTER MAPPING PRODUCTION
# ============================================================

print("=" * 70)
print("FULL DETECTION → LOCAL-CLUSTER MAPPING PRODUCTION")
print("=" * 70)

mapping_files = sorted(
    MAPPING_DIR.glob(
        "detection_cluster_mapping_*.parquet"
    )
)

existing_mapping_dates = {
    pd.Timestamp(
        p.stem.replace(
            "detection_cluster_mapping_",
            ""
        )
    ).date()
    for p in mapping_files
}

completed = 0
skipped = 0
failed = 0

for i, date_value in enumerate(
    observed_date_values,
    start=1
):

    mapping_path = (
        MAPPING_DIR
        / f"detection_cluster_mapping_{date_value}.parquet"
    )

    if date_value in existing_mapping_dates:
        skipped += 1
        continue

    print("\n" + "-" * 70)
    print(
        f"[{i}/{len(observed_date_values)}] "
        f"PROCESSING {date_value}"
    )
    print("-" * 70)

    try:

        day_df = df.loc[
            date_index[date_value]
        ].copy()

        work_day, summary_day, diagnostics = (
            cluster_single_day(
                day_df
            )
        )

        mapping = work_day[
            [
                "_original_index",
                "local_cluster_id"
            ]
        ].copy()

        mapping = mapping.rename(
            columns={
                "_original_index":
                    "detection_index"
            }
        )

        # ----------------------------------------------------
        # INTEGRITY CHECK
        # ----------------------------------------------------

        assert len(mapping) == len(day_df)

        assert (
            mapping["detection_index"]
            .nunique()
            == len(day_df)
        )

        assert (
            mapping["local_cluster_id"]
            .notna()
            .all()
        )

        assert (
            mapping["local_cluster_id"]
            .nunique()
            == diagnostics["n_clusters"]
        )

        # ----------------------------------------------------
        # SAVE
        # ----------------------------------------------------

        mapping.to_parquet(
            mapping_path,
            index=False
        )

        completed += 1

        print(
            "Detections:",
            len(day_df)
        )

        print(
            "Local clusters:",
            diagnostics["n_clusters"]
        )

        print(
            "Saved:",
            mapping_path.name
        )

    except Exception as e:

        failed += 1

        print(
            f"❌ FAILED: {date_value}"
        )

        print(
            repr(e)
        )

        raise

print("\n" + "=" * 70)
print("MAPPING PRODUCTION COMPLETE")
print("=" * 70)

print("New mappings :", completed)
print("Already exist:", skipped)
print("Failed       :", failed)


Streaming output truncated to the last 5000 lines.
Local clusters: 338
Saved: detection_cluster_mapping_2024-09-17.parquet

----------------------------------------------------------------------
[4625/5337] PROCESSING 2024-09-18
----------------------------------------------------------------------
Detections: 1102
Local clusters: 422
Saved: detection_cluster_mapping_2024-09-18.parquet

----------------------------------------------------------------------
[4626/5337] PROCESSING 2024-09-19
----------------------------------------------------------------------
Detections: 995
Local clusters: 414
Saved: detection_cluster_mapping_2024-09-19.parquet

----------------------------------------------------------------------
[4627/5337] PROCESSING 2024-09-20
----------------------------------------------------------------------
Detections: 861
Local clusters: 417
Saved: detection_cluster_mapping_2024-09-20.parquet

----------------------------------------------------------------------
[4628/533